# Cleaning3
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

client = DatalakeClient()

# Get the files 
Exclusevily from ADNI dataset stored in the Datalake

In [ ]:
file_codes = ['UCSFFSX']#'ADNIMERGE', 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 'BLCHANGE', 'DXSUM', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T']

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_02', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

## Operazioni
- Trasformare i volumi come percentuali di ICV

In [4]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned2'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned3'

In [10]:
if os.path.isfile(new_name+'.xlsx'):
    update_new_support_file(support_file, new_name)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)

The ADNI_variables_cleaned3 file has been created.
Open the file and verify it, if needed update the variables names and metadata


In [13]:
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')
new_support_file = pd.read_excel(new_name+'.xlsx')

In [ ]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    #new_support_file = dataCleaner.update_self_support_file(new_support_file)
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(df_new, file_code)
        # Transform volumes as ICV percentage
        final_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    else:
        final_df = df_new
    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_03', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)
    print('updated_metadata: ', updated_metadata)
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)

    # Create new file name for datalake
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_03')


    # upload the new file
    #result = client.upload_dataframe(
        #df=final_df,
        #object_name=new_file_name,
        #prefix='cleaned/single_file/',
        #metadata=updated_metadata
    #)
   
    
#save_df(df_to_save=new_support_file, output_path=new_name)     



 ---- UCSFFSX_11_02_15_11Aug2025_02.csv
label:  MidTemp
Sommo i volumi R e L per la label MidTemp
label:  Hippocampus
Sommo i volumi R e L per la label Hippocampus
label:  Entorhinal
Sommo i volumi R e L per la label Entorhinal
label:  Ventricle
Sommo i volumi R e L per la label Ventricle
label:  Fusiform
Sommo i volumi R e L per la label Fusiform
norm_volume ['ICV%ICV', 'MidTemp%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Ventricle%ICV', 'Fusiform%ICV']
entro in volume
volume_metadata {}
updated_metadata:  {'cofattori': [], 'file_code': 'UCSFFSX', 'level': 'cleaned_03', 'norm_intervallo': [], 'norm_scala': [], 'norm_scale_value': [], 'norm_volume': ['ICV%ICV', 'MidTemp%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Ventricle%ICV', 'Fusiform%ICV'], 'population': ['ADNI1', 'ADNIGO', 'ADNI2'], 'predittori': ['ICV%ICV', 'MidTemp%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Ventricle%ICV', 'Fusiform%ICV'], 'source': 'ADNI', 'volume_norm_values': {}}


In [ ]:
final_df